In [4]:
from pathlib import Path
import polars as pl
from loguru import logger

BASE_DIR = Path("..")
SILVER_DIR = BASE_DIR / "data" / "2_silver"
GOLD_DIR = BASE_DIR / "data" / "3_gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

# Nạp dữ liệu tầng Silver
df_0910_clean = pl.read_parquet(SILVER_DIR / "sheet_2009_2010_cleaned.parquet")
df_1011_clean = pl.read_parquet(SILVER_DIR / "sheet_2010_2011_cleaned.parquet")

logger.info(f"Sheet 09-10 Silver: {len(df_0910_clean):,} dòng")
logger.info(f"Sheet 10-11 Silver: {len(df_1011_clean):,} dòng")

2026-11-04 16:45:52.975 | INFO     | __main__:<module>:14 - Sheet 09-10 Silver: 511,565 dòng
2026-11-04 16:45:52.976 | INFO     | __main__:<module>:15 - Sheet 10-11 Silver: 530,105 dòng


Gộp 2 Sheet & Gom nhóm theo Ngày x Sản phẩm 

In [5]:
# 1. Gộp 2 DataFrame liên tục
df_combined = pl.concat([df_0910_clean, df_1011_clean])
logger.info(f"Tổng dữ liệu sau khi gộp 2 sheet: {len(df_combined):,} dòng giao dịch.")

# 2. Tạo cột Ngày (bỏ giờ phút giây) và tính Doanh thu từng dòng
df_combined = df_combined.with_columns([
    pl.col("InvoiceDate").dt.date().alias("Date"),
    (pl.col("Quantity") * pl.col("UnitPrice")).alias("LineTotal")
])

# 3. Aggregation: Tổng hợp Demand và Doanh thu theo (Date, StockCode)
df_daily = df_combined.group_by(["Date", "StockCode"]).agg([
    pl.col("Quantity").sum().alias("Daily_Demand"),
    pl.col("LineTotal").sum().alias("Daily_Revenue"),
    pl.col("UnitPrice").mean().alias("Avg_UnitPrice"),
    pl.col("Description").first().alias("Description")
]).sort(["StockCode", "Date"])

print(f"Số lượng bản ghi sau khi gom nhóm theo Ngày x Sản phẩm: {len(df_daily):,} dòng")
display(df_daily.head(5))

2026-11-04 16:45:52.994 | INFO     | __main__:<module>:3 - Tổng dữ liệu sau khi gộp 2 sheet: 1,041,670 dòng giao dịch.


Số lượng bản ghi sau khi gom nhóm theo Ngày x Sản phẩm: 532,838 dòng


Date,StockCode,Daily_Demand,Daily_Revenue,Avg_UnitPrice,Description
date,str,i64,f64,f64,str
2009-12-01,"""10002""",12,10.2,0.85,"""INFLATABLE POLITICAL GLOBE"""
2009-12-03,"""10002""",7,5.95,0.85,"""INFLATABLE POLITICAL GLOBE"""
2009-12-04,"""10002""",73,62.05,0.85,"""INFLATABLE POLITICAL GLOBE"""
2009-12-06,"""10002""",49,41.65,0.85,"""INFLATABLE POLITICAL GLOBE"""
2009-12-07,"""10002""",2,1.7,0.85,"""INFLATABLE POLITICAL GLOBE"""


Zero-Filling (Điền 0 cho những ngày sản phẩm không có giao dịch)

In [6]:
# Tạo lưới đầy đủ tất cả các Ngày x các Mã Sản phẩm (Grid of Date x StockCode)
unique_dates = df_daily.select("Date").unique().sort("Date")
unique_stocks = df_daily.select("StockCode").unique()

# Cross join để tạo tất cả các cặp có thể có
grid = unique_dates.join(unique_stocks, how="cross")

# Join dữ liệu bán hàng thực tế vào lưới, ngày nào thiếu thì fill = 0
df_gold_full = grid.join(df_daily, on=["Date", "StockCode"], how="left").with_columns([
    pl.col("Daily_Demand").fill_null(0),
    pl.col("Daily_Revenue").fill_null(0.0),
    pl.col("Avg_UnitPrice").forward_fill().over("StockCode").fill_null(0.0),
    pl.col("Description").forward_fill().over("StockCode").fill_null("UNKNOWN")
]).sort(["StockCode", "Date"])

logger.success(f" Đã Zero-Filling thành công! Tổng số bản ghi chuỗi thời gian hoàn chỉnh: {len(df_gold_full):,} dòng")

2026-11-04 16:46:40.762 | SUCCESS  | __main__:<module>:16 -  Đã Zero-Filling thành công! Tổng số bản ghi chuỗi thời gian hoàn chỉnh: 2,969,264 dòng


In [10]:
df_gold_feat = df_gold_full.with_columns([
    pl.col("Date").dt.day().alias("Day"),
    pl.col("Date").dt.weekday().alias("DayOfWeek"),
    pl.col("Date").dt.month().alias("Month"),
    pl.col("Date").dt.quarter().alias("Quarter"),
    pl.col("Date").dt.year().alias("Year"),
    (pl.col("Date").dt.weekday() >= 6).cast(pl.Int32).alias("Is_Weekend")
])

logger.success("Đã trích xuất xong các đặc trưng thời gian (Calendar Features).")

2026-11-04 16:47:34.961 | SUCCESS  | __main__:<module>:10 - Đã trích xuất xong các đặc trưng thời gian (Calendar Features).


Tạo Đặc trưng Trễ (Lag Features) & Trung bình trượt (Rolling Features)

In [11]:
# Tạo Lag Features (Demand của 1 ngày, 7 ngày, 14 ngày trước) và Rolling Mean 7 ngày
df_gold_feat = df_gold_feat.with_columns([
    pl.col("Daily_Demand").shift(1).over("StockCode").alias("Lag_1"),
    pl.col("Daily_Demand").shift(7).over("StockCode").alias("Lag_7"),
    pl.col("Daily_Demand").shift(14).over("StockCode").alias("Lag_14"),
    
    # Target: Dự báo nhu cầu 7 ngày tới (Shift -7)
    pl.col("Daily_Demand").shift(-7).over("StockCode").alias("Target_Demand_7D"),
    
    # Rolling Mean 7 ngày qua
    pl.col("Daily_Demand").rolling_mean(window_size=7).over("StockCode").alias("Rolling_Mean_7D")
])

# Loại bỏ các dòng bị NULL do phép trễ (shift) tạo ra ở đầu/cuối chuỗi
df_gold_final = df_gold_feat.filter(
    pl.col("Lag_14").is_not_null() & pl.col("Target_Demand_7D").is_not_null()
)

logger.success(f"Tạo Lag & Rolling Features hoàn tất! Số dòng huấn luyện sẵn sàng: {len(df_gold_final):,} dòng")

2026-11-04 16:47:44.413 | SUCCESS  | __main__:<module>:19 - Tạo Lag & Rolling Features hoàn tất! Số dòng huấn luyện sẵn sàng: 2,866,028 dòng


In [ ]:
display(df_gold_final.head(50))

Date,StockCode,Daily_Demand,Daily_Revenue,Avg_UnitPrice,Description,Day,DayOfWeek,Month,Quarter,Year,Is_Weekend,Lag_1,Lag_7,Lag_14,Target_Demand_7D,Rolling_Mean_7D
date,str,i64,f64,f64,str,i8,i8,i8,i8,i32,i32,i64,i64,i64,i64,f64
2009-12-16,"""10002""",0,0.0,1.133333,"""INFLATABLE POLITICAL GLOBE""",16,3,12,4,2009,0,0,12,12,3,6.714286
2009-12-17,"""10002""",0,0.0,1.133333,"""INFLATABLE POLITICAL GLOBE""",17,4,12,4,2009,0,0,0,0,0,6.714286
2009-12-18,"""10002""",0,0.0,1.133333,"""INFLATABLE POLITICAL GLOBE""",18,5,12,4,2009,0,0,1,7,0,6.571429
2009-12-20,"""10002""",0,0.0,1.133333,"""INFLATABLE POLITICAL GLOBE""",20,7,12,4,2009,1,0,9,73,0,5.285714
2009-12-21,"""10002""",12,10.2,0.85,"""INFLATABLE POLITICAL GLOBE""",21,1,12,4,2009,0,0,0,0,12,7.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2010-02-17,"""10002""",12,10.2,0.85,"""INFLATABLE POLITICAL GLOBE""",17,3,2,1,2010,0,1,12,4,112,6.142857
2010-02-18,"""10002""",24,20.4,0.85,"""INFLATABLE POLITICAL GLOBE""",18,4,2,1,2010,0,12,0,0,0,9.571429
2010-02-19,"""10002""",0,0.0,0.85,"""INFLATABLE POLITICAL GLOBE""",19,5,2,1,2010,0,24,30,12,1,5.285714
